# 🎯 Complete Hyperparameter Tuning Masterclass
### Every Classifier & Regressor in Supervised ML — Explained + Tuned

This notebook covers **every hyperparameter** for **18 algorithms**:

**📈 Regression (8):** Linear, Polynomial, Ridge, Lasso, Decision Tree, Random Forest, SVR, KNN

**🏷️ Classification (10):** Logistic Regression, KNN, SVC, Decision Tree, Random Forest, Extra Trees, AdaBoost, Naive Bayes (Gaussian/Multinomial/Bernoulli)

For every algorithm you get:
- ✅ What each hyperparameter does & **why** you'd tune it
- ✅ A working `GridSearchCV` / `RandomizedSearchCV` example
- ✅ Best practice tips

---


## 🔧 0. Setup — Imports & Datasets
We use `sklearn`'s built-in datasets: a regression dataset (California Housing) and a classification dataset (Breast Cancer) so every example below is runnable end-to-end.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, KFold, StratifiedKFold)
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                              classification_report)

# ----- Regression dataset -----
housing = fetch_california_housing()
Xr = pd.DataFrame(housing.data, columns=housing.feature_names)
yr = housing.target
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
Xr_train_s = scaler_r.fit_transform(Xr_train)
Xr_test_s  = scaler_r.transform(Xr_test)

# ----- Classification dataset -----
cancer = load_breast_cancer()
Xc = pd.DataFrame(cancer.data, columns=cancer.feature_names)
yc = cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

scaler_c = StandardScaler()
Xc_train_s = scaler_c.fit_transform(Xc_train)
Xc_test_s  = scaler_c.transform(Xc_test)

print("Regression  :", Xr_train.shape, Xr_test.shape)
print("Classification:", Xc_train.shape, Xc_test.shape)


Regression  : (16512, 8) (4128, 8)
Classification: (455, 30) (114, 30)


## 🧠 Quick Primer: What *is* Hyperparameter Tuning?

| Term | Meaning |
|---|---|
| **Parameter** | Learned automatically from data during training (e.g. coefficients of a linear model) |
| **Hyperparameter** | Set *before* training; controls how the model learns (e.g. `max_depth`, `C`, `k`) |
| **GridSearchCV** | Exhaustively tries every combination of given hyperparameter values, using cross-validation |
| **RandomizedSearchCV** | Samples random combinations — much faster for large search spaces |
| **Cross-Validation (CV)** | Splits training data into *k* folds to get a reliable estimate of a hyperparameter combo's performance |

**Why tune?** Default hyperparameters are rarely optimal for your specific dataset. Tuning trades off **bias vs variance** (underfitting vs overfitting) to get the best generalization.


# 📈 PART 1 — REGRESSION ALGORITHMS

## 1️⃣ Linear Regression

Plain Linear Regression has **almost no hyperparameters** to tune (no regularization term) — it's included here mainly as a baseline, but it does have a few settings worth knowing:

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `fit_intercept` | Whether to calculate the intercept (bias) term | Set `False` only if data is already centered at 0 |
| `positive` | Forces all coefficients to be positive | Useful when domain knowledge says relationships must be positive (e.g. price ↑ with size ↑) |
| `copy_X` | Whether X is copied before fitting | Rarely changed — memory optimization only |

Because there's nothing to "search" meaningfully, we just fit it directly as our baseline.

In [3]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression(fit_intercept=True, positive=False)
lr.fit(Xr_train_s, yr_train)
pred = lr.predict(Xr_test_s)
print("Linear Regression  R2:", r2_score(yr_test, pred), " RMSE:", mean_squared_error(yr_test, pred))


Linear Regression  R2: 0.575787706032451  RMSE: 0.5558915986952442


## 2️⃣ Polynomial Regression

This is Linear Regression applied to **polynomial-expanded features**. The real hyperparameter here is the **degree** of the polynomial (plus, optionally, regularization if combined with Ridge/Lasso).

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `degree` | Highest power of features generated (x, x², x³...) | Low degree → underfits (too simple). High degree → overfits (wiggly curve that memorizes noise) |
| `interaction_only` | If True, only interaction terms (x1*x2) — no powers (x1²) | Use when you only care about feature interactions, not curvature |
| `include_bias` | Adds a constant (bias) column of 1s | Usually left True; set False if your downstream model already fits an intercept |

We tune `degree` with GridSearchCV inside a `Pipeline`.

In [ ]:
from sklearn.pipeline import Pipeline

poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scale', StandardScaler()),
    ('lin', LinearRegression())
])

param_grid_poly = {
    'poly__degree': [1, 2, 3, 4],          # try linear up to degree-4 curve
    'poly__interaction_only': [False, True]
}

# Use a SMALL subset of features for speed (polynomial blows up dimensionality fast)
Xr_small = Xr_train[['MedInc', 'HouseAge', 'AveRooms']]
Xr_small_test = Xr_test[['MedInc', 'HouseAge', 'AveRooms']]

grid_poly = GridSearchCV(poly_pipe, param_grid_poly, cv=5,
                          scoring='r2', n_jobs=-1)
grid_poly.fit(Xr_small, yr_train)

print("Best params:", grid_poly.best_params_)
print("Best CV R2 :", grid_poly.best_score_)
print("Test R2    :", grid_poly.score(Xr_small_test, yr_test))


## 3️⃣ Ridge Regression (L2 Regularization)

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `alpha` | Strength of L2 penalty on coefficients | **Most important.** Higher `alpha` → smaller coefficients → less overfitting but more bias. Low `alpha` ≈ plain Linear Regression |
| `solver` | Algorithm used to solve the optimization (`auto`, `svd`, `cholesky`, `lsqr`, `saga`...) | `saga`/`sag` scale better to huge datasets; `svd` is more numerically stable for small/collinear data |
| `fit_intercept` | Whether to fit the bias term | Keep True unless data pre-centered |
| `max_iter` | Max iterations for iterative solvers | Increase if solver doesn't converge (warning raised) |
| `tol` | Convergence tolerance | Smaller = more precise but slower |

In [ ]:
from sklearn.linear_model import Ridge

param_grid_ridge = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr'],
}

grid_ridge = GridSearchCV(Ridge(), param_grid_ridge, cv=5, scoring='r2', n_jobs=-1)
grid_ridge.fit(Xr_train_s, yr_train)

print("Best params:", grid_ridge.best_params_)
print("Best CV R2 :", grid_ridge.best_score_)
print("Test R2    :", grid_ridge.score(Xr_test_s, yr_test))


## 4️⃣ Lasso Regression (L1 Regularization)

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `alpha` | Strength of L1 penalty | **Key parameter.** Larger `alpha` forces more coefficients to exactly **0** → automatic feature selection. Too large → underfits (everything zeroed out) |
| `max_iter` | Max iterations of coordinate descent | Lasso often needs more iterations than Ridge to converge — raise if you see a `ConvergenceWarning` |
| `selection` | `cyclic` or `random` feature update order | `random` can converge faster on large, sparse datasets |
| `tol` | Convergence tolerance | Lower = more precise, slower |

In [ ]:
from sklearn.linear_model import Lasso

param_grid_lasso = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'selection': ['cyclic', 'random'],
    'max_iter': [5000]
}

grid_lasso = GridSearchCV(Lasso(), param_grid_lasso, cv=5, scoring='r2', n_jobs=-1)
grid_lasso.fit(Xr_train_s, yr_train)

print("Best params:", grid_lasso.best_params_)
print("Best CV R2 :", grid_lasso.best_score_)
print("Test R2    :", grid_lasso.score(Xr_test_s, yr_test))

# bonus: see which features got zeroed out
coef_df = pd.Series(grid_lasso.best_estimator_.coef_, index=Xr.columns)
print("\nFeatures zeroed by Lasso:", list(coef_df[coef_df == 0].index))


## 5️⃣ Decision Tree Regressor

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `max_depth` | Max depth of the tree | **Most important.** Deep tree → overfits (memorizes). Shallow tree → underfits |
| `min_samples_split` | Min samples required to split an internal node | Higher value prevents splits on tiny, noisy subsets |
| `min_samples_leaf` | Min samples required at a leaf node | Smooths the model; prevents leaves that represent 1 outlier |
| `max_features` | Number/fraction of features considered at each split | Reduces correlation between splits, adds randomness, can reduce overfitting |
| `criterion` | Split quality measure (`squared_error`, `friedman_mse`, `absolute_error`, `poisson`) | `absolute_error` is robust to outliers but much slower |
| `ccp_alpha` | Complexity parameter for **cost-complexity pruning** | Larger value → more aggressive pruning → simpler tree |

In [ ]:
from sklearn.tree import DecisionTreeRegressor

param_grid_dtr = {
    'max_depth': [3, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': [None, 'sqrt', 'log2'],
    'criterion': ['squared_error', 'friedman_mse']
}

random_dtr = RandomizedSearchCV(DecisionTreeRegressor(random_state=42), param_grid_dtr,
                                 n_iter=25, cv=5, scoring='r2', n_jobs=-1, random_state=42)
random_dtr.fit(Xr_train, yr_train)  # trees don't need scaling

print("Best params:", random_dtr.best_params_)
print("Best CV R2 :", random_dtr.best_score_)
print("Test R2    :", random_dtr.score(Xr_test, yr_test))


## 6️⃣ Random Forest Regressor

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_estimators` | Number of trees in the forest | More trees → more stable predictions, but diminishing returns + slower |
| `max_depth` | Max depth of each tree | Controls individual tree overfitting |
| `min_samples_split` / `min_samples_leaf` | Same as Decision Tree, applied per tree | Controls overfitting per tree |
| `max_features` | Features considered per split (`sqrt`, `log2`, fraction) | **Key for randomness** — lower value = more diverse (decorrelated) trees = less overfitting |
| `bootstrap` | Whether to sample with replacement to build each tree | True = classic Random Forest (bagging); False = uses whole dataset each time |
| `max_samples` | Fraction of samples drawn for each tree (if bootstrap=True) | Lower value → faster training, more regularization |

In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rfr = {
    'n_estimators': [100, 200, 400],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2', 1.0],
    'bootstrap': [True]
}

random_rfr = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid_rfr,
                                 n_iter=20, cv=3, scoring='r2', n_jobs=-1, random_state=42)
random_rfr.fit(Xr_train, yr_train)

print("Best params:", random_rfr.best_params_)
print("Best CV R2 :", random_rfr.best_score_)
print("Test R2    :", random_rfr.score(Xr_test, yr_test))


## 7️⃣ Support Vector Regression (SVR)

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `C` | Regularization strength (inverse) | High `C` → fits training data tightly (low bias, risk overfit). Low `C` → smoother, more regularized |
| `epsilon` | Width of the "no-penalty" tube around predictions | Larger epsilon → fewer support vectors, simpler/flatter model |
| `kernel` | Transforms data into higher dimensions (`linear`, `poly`, `rbf`, `sigmoid`) | `rbf` handles non-linear relationships; `linear` is faster & more interpretable |
| `gamma` | Kernel coefficient for `rbf`/`poly`/`sigmoid` | High gamma → each point has small influence radius → can overfit. Low gamma → smoother decision surface |
| `degree` | Degree of the `poly` kernel only | Higher degree = more complex curve |

⚠️ SVR is scale-sensitive — **always scale your features** first.

In [ ]:
from sklearn.svm import SVR

param_grid_svr = {
    'C': [0.1, 1, 10, 100],
    'epsilon': [0.01, 0.1, 0.5],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

random_svr = RandomizedSearchCV(SVR(), param_grid_svr, n_iter=15, cv=5,
                                 scoring='r2', n_jobs=-1, random_state=42)
# subsample for speed - SVR is O(n^2)/O(n^3)
sub_idx = np.random.RandomState(42).choice(len(Xr_train_s), 3000, replace=False)
random_svr.fit(Xr_train_s[sub_idx], yr_train.values[sub_idx])

print("Best params:", random_svr.best_params_)
print("Best CV R2 :", random_svr.best_score_)
print("Test R2    :", random_svr.score(Xr_test_s, yr_test))


## 8️⃣ K-Nearest Neighbors Regressor

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_neighbors` (k) | Number of neighbors averaged for prediction | **Most important.** Small k → overfits (noisy, follows every point). Large k → underfits (over-smoothed) |
| `weights` | `uniform` (equal vote) vs `distance` (closer points weigh more) | `distance` usually helps when nearby points matter more |
| `metric` | Distance function (`minkowski`, `euclidean`, `manhattan`) | Choice depends on feature geometry / scale |
| `p` | Power parameter for Minkowski metric (p=1 → Manhattan, p=2 → Euclidean) | Manhattan is more robust in high dimensions |
| `algorithm` | Search algorithm (`auto`, `ball_tree`, `kd_tree`, `brute`) | Affects speed only, not accuracy |

⚠️ KNN is distance-based — **always scale features**.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

param_grid_knnr = {
    'n_neighbors': [3, 5, 7, 10, 15, 20],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

grid_knnr = GridSearchCV(KNeighborsRegressor(), param_grid_knnr, cv=5, scoring='r2', n_jobs=-1)
grid_knnr.fit(Xr_train_s, yr_train)

print("Best params:", grid_knnr.best_params_)
print("Best CV R2 :", grid_knnr.best_score_)
print("Test R2    :", grid_knnr.score(Xr_test_s, yr_test))


### 📊 Regression Summary

| Model | Most impactful hyperparameter | Typical risk if mis-tuned |
|---|---|---|
| Linear Regression | `fit_intercept` | Minimal risk — few knobs |
| Polynomial Regression | `degree` | Overfitting at high degree |
| Ridge | `alpha` | Over/under-shrinkage |
| Lasso | `alpha` | Dropping important features |
| Decision Tree | `max_depth` | Overfitting (deep) / underfitting (shallow) |
| Random Forest | `n_estimators`, `max_features` | Slow training / too-correlated trees |
| SVR | `C`, `gamma` | Overfitting (high C/gamma) |
| KNN | `n_neighbors` | Noisy (low k) / over-smoothed (high k) |


# 🏷️ PART 2 — CLASSIFICATION ALGORITHMS

## 1️⃣ Logistic Regression

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `C` | Inverse of regularization strength | Low `C` → stronger regularization → simpler decision boundary. High `C` → fits training data closely |
| `penalty` | Type of regularization (`l1`, `l2`, `elasticnet`, `none`) | `l1` does feature selection (sparsity); `l2` shrinks smoothly; `elasticnet` mixes both |
| `solver` | Optimization algorithm (`lbfgs`, `liblinear`, `saga`, `newton-cg`) | Must match penalty: `liblinear`/`saga` support `l1`; `lbfgs` only supports `l2`/`none` |
| `l1_ratio` | Mix ratio between L1/L2 (only for `elasticnet`) | Tune between 0 (pure L2) and 1 (pure L1) |
| `max_iter` | Max optimization iterations | Raise if convergence warning appears |
| `class_weight` | Weighting applied per class (`balanced` or custom dict) | Use `balanced` for imbalanced classes so minority class isn't ignored |

In [ ]:
from sklearn.linear_model import LogisticRegression

param_grid_logreg = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],         # liblinear supports both l1 & l2, good for small datasets
    'class_weight': [None, 'balanced']
}

grid_logreg = GridSearchCV(LogisticRegression(max_iter=5000), param_grid_logreg,
                            cv=5, scoring='accuracy', n_jobs=-1)
grid_logreg.fit(Xc_train_s, yc_train)

print("Best params:", grid_logreg.best_params_)
print("Best CV Acc:", grid_logreg.best_score_)
print("Test Acc   :", grid_logreg.score(Xc_test_s, yc_test))


## 2️⃣ K-Nearest Neighbors (KNN) Classifier

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_neighbors` (k) | Number of neighbors that vote on the class | Small k → overfits/noisy boundary. Large k → underfits, biased toward majority class |
| `weights` | `uniform` vs `distance` | `distance` helps when class boundaries are irregular |
| `metric` | Distance function | `manhattan` can outperform `euclidean` in high dimensions |
| `p` | Minkowski power (1=Manhattan, 2=Euclidean) | Affects how "distance" is measured |
| `algorithm` | `ball_tree`/`kd_tree`/`brute`/`auto` | Speed only |

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

param_grid_knnc = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

grid_knnc = GridSearchCV(KNeighborsClassifier(), param_grid_knnc, cv=5, scoring='accuracy', n_jobs=-1)
grid_knnc.fit(Xc_train_s, yc_train)

print("Best params:", grid_knnc.best_params_)
print("Best CV Acc:", grid_knnc.best_score_)
print("Test Acc   :", grid_knnc.score(Xc_test_s, yc_test))


## 3️⃣ Support Vector Classifier (SVC)

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `C` | Regularization (margin softness) | High `C` → narrow margin, fits training points tightly (overfit risk). Low `C` → wider margin, more tolerant of misclassification |
| `kernel` | `linear`, `poly`, `rbf`, `sigmoid` | `rbf` handles non-linear boundaries; `linear` for high-dim/sparse/text data |
| `gamma` | Influence radius of a single training example (`rbf`/`poly`/`sigmoid`) | High gamma → tight, wiggly boundary (overfit). Low gamma → smooth boundary (underfit) |
| `degree` | Degree of `poly` kernel | Higher = more complex curved boundary |
| `class_weight` | Per-class weighting | `balanced` helps with imbalanced datasets |

In [ ]:
from sklearn.svm import SVC

param_grid_svc = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced']
}

grid_svc = GridSearchCV(SVC(), param_grid_svc, cv=5, scoring='accuracy', n_jobs=-1)
grid_svc.fit(Xc_train_s, yc_train)

print("Best params:", grid_svc.best_params_)
print("Best CV Acc:", grid_svc.best_score_)
print("Test Acc   :", grid_svc.score(Xc_test_s, yc_test))


## 4️⃣ Decision Tree Classifier

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `max_depth` | Max depth of tree | Deep → overfits; shallow → underfits |
| `min_samples_split` | Min samples to split a node | Prevents splitting on tiny/noisy subsets |
| `min_samples_leaf` | Min samples at a leaf | Smooths predictions, avoids 1-sample leaves |
| `max_features` | Features considered per split | Adds randomness, can reduce overfitting |
| `criterion` | `gini` or `entropy` (split quality) | Usually similar results; `entropy` is slightly more computationally expensive |
| `class_weight` | Per-class weighting | `balanced` for imbalanced classes |
| `ccp_alpha` | Cost-complexity pruning strength | Higher = simpler, more pruned tree |

In [ ]:
from sklearn.tree import DecisionTreeClassifier

param_grid_dtc = {
    'max_depth': [3, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

random_dtc = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dtc,
                                 n_iter=25, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
random_dtc.fit(Xc_train, yc_train)

print("Best params:", random_dtc.best_params_)
print("Best CV Acc:", random_dtc.best_score_)
print("Test Acc   :", random_dtc.score(Xc_test, yc_test))


## 5️⃣ Random Forest Classifier

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_estimators` | Number of trees | More trees → more stable, but diminishing returns + slower |
| `max_depth` | Depth of each tree | Controls overfitting per tree |
| `min_samples_split`/`min_samples_leaf` | Same as single tree | Controls overfitting |
| `max_features` | Features per split | Core "randomness" knob — decorrelates trees |
| `bootstrap` | Sampling with replacement | True = classic bagging |
| `class_weight` | Per-class weighting | `balanced`/`balanced_subsample` for imbalance |

In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rfc = {
    'n_estimators': [100, 200, 400],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}

random_rfc = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_rfc,
                                 n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)
random_rfc.fit(Xc_train, yc_train)

print("Best params:", random_rfc.best_params_)
print("Best CV Acc:", random_rfc.best_score_)
print("Test Acc   :", random_rfc.score(Xc_test, yc_test))


## 6️⃣ Extra Trees Classifier (Extremely Randomized Trees)

Like Random Forest, but splits are chosen **randomly** (not the best split) — adds even more randomness, usually faster to train.

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_estimators` | Number of trees | More trees = more stable predictions |
| `max_depth` | Tree depth | Controls overfitting |
| `min_samples_split`/`min_samples_leaf` | Node splitting thresholds | Controls overfitting |
| `max_features` | Features considered per split | Controls how random each tree is |
| `bootstrap` | Whether to bootstrap samples (default **False** in Extra Trees, unlike RF) | `False` = each tree sees the full dataset (more randomness comes from split selection instead) |

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

param_grid_etc = {
    'n_estimators': [100, 200, 400],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [False, True]
}

random_etc = RandomizedSearchCV(ExtraTreesClassifier(random_state=42, n_jobs=-1), param_grid_etc,
                                 n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)
random_etc.fit(Xc_train, yc_train)

print("Best params:", random_etc.best_params_)
print("Best CV Acc:", random_etc.best_score_)
print("Test Acc   :", random_etc.score(Xc_test, yc_test))


## 7️⃣ AdaBoost Classifier

Boosting: sequentially trains weak learners (default: shallow Decision Trees), each one focusing more on previously misclassified points.

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `n_estimators` | Number of weak learners (boosting rounds) | More rounds → more complex model; too many can overfit |
| `learning_rate` | Shrinks the contribution of each weak learner | Lower learning rate + more estimators = more robust (classic bias/variance trade-off); typically tuned **together** with `n_estimators` |
| `estimator` (base learner) | The weak learner used (default: depth-1 `DecisionTreeClassifier`) | A slightly deeper base tree can capture more complex patterns per round |
| `algorithm` | `SAMME` (discrete boosting algorithm) | Affects how sample weights are updated |

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

param_grid_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'estimator': [DecisionTreeClassifier(max_depth=1), DecisionTreeClassifier(max_depth=2)]
}

grid_ada = GridSearchCV(AdaBoostClassifier(random_state=42), param_grid_ada,
                         cv=5, scoring='accuracy', n_jobs=-1)
grid_ada.fit(Xc_train, yc_train)

print("Best params:", grid_ada.best_params_)
print("Best CV Acc:", grid_ada.best_score_)
print("Test Acc   :", grid_ada.score(Xc_test, yc_test))


## 8️⃣–🔟 Naive Bayes Family

Naive Bayes models have **very few hyperparameters** because they're based on fixed probability assumptions rather than iterative optimization — but the few they have matter a lot.

### 8️⃣ Gaussian Naive Bayes
Assumes continuous features follow a normal distribution within each class. Best for continuous, roughly-normal data.

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `var_smoothing` | Adds a small fraction of the largest feature variance to all variances | Prevents division-by-zero / numerical instability when a feature has near-zero variance; also acts as mild regularization |

### 9️⃣ Multinomial Naive Bayes
Best for **count data** (e.g. word counts in text classification). Requires non-negative features.

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `alpha` | Laplace/Lidstone smoothing constant | Prevents zero probability for unseen feature values in a class; higher `alpha` = more smoothing = less overfitting to rare words/features |
| `fit_prior` | Whether to learn class prior probabilities from data | Set `False` to force a uniform prior (equal class probability) regardless of class imbalance |

### 🔟 Bernoulli Naive Bayes
Best for **binary/boolean features** (e.g. word presence/absence).

| Hyperparameter | What it does | Why tune it |
|---|---|---|
| `alpha` | Laplace smoothing constant | Same role as in Multinomial NB |
| `binarize` | Threshold to convert features to 0/1 | Needs tuning if your features aren't already binary — this threshold decides the cutoff |
| `fit_prior` | Learn priors from data vs uniform | Same role as above |

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.preprocessing import MinMaxScaler

# ---- Gaussian NB ----
param_grid_gnb = {'var_smoothing': np.logspace(-11, -5, 7)}
grid_gnb = GridSearchCV(GaussianNB(), param_grid_gnb, cv=5, scoring='accuracy', n_jobs=-1)
grid_gnb.fit(Xc_train_s, yc_train)
print("GaussianNB  -> best params:", grid_gnb.best_params_,
      "| CV Acc:", round(grid_gnb.best_score_, 4),
      "| Test Acc:", round(grid_gnb.score(Xc_test_s, yc_test), 4))

# ---- Multinomial NB (needs non-negative features -> MinMax scale to [0,1]) ----
mm = MinMaxScaler()
Xc_train_mm = mm.fit_transform(Xc_train)
Xc_test_mm  = mm.transform(Xc_test)

param_grid_mnb = {'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0], 'fit_prior': [True, False]}
grid_mnb = GridSearchCV(MultinomialNB(), param_grid_mnb, cv=5, scoring='accuracy', n_jobs=-1)
grid_mnb.fit(Xc_train_mm, yc_train)
print("MultinomialNB -> best params:", grid_mnb.best_params_,
      "| CV Acc:", round(grid_mnb.best_score_, 4),
      "| Test Acc:", round(grid_mnb.score(Xc_test_mm, yc_test), 4))

# ---- Bernoulli NB ----
param_grid_bnb = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0],
    'binarize': [0.0, 0.5, 1.0, mm.data_max_.mean()/2],
    'fit_prior': [True, False]
}
grid_bnb = GridSearchCV(BernoulliNB(), param_grid_bnb, cv=5, scoring='accuracy', n_jobs=-1)
grid_bnb.fit(Xc_train_mm, yc_train)
print("BernoulliNB -> best params:", grid_bnb.best_params_,
      "| CV Acc:", round(grid_bnb.best_score_, 4),
      "| Test Acc:", round(grid_bnb.score(Xc_test_mm, yc_test), 4))


### 📊 Classification Summary

| Model | Most impactful hyperparameter | Typical risk if mis-tuned |
|---|---|---|
| Logistic Regression | `C`, `penalty` | Over-regularized (underfit) / overfit |
| KNN | `n_neighbors` | Noisy boundary (low k) / underfit (high k) |
| SVC | `C`, `gamma` | Overfitting with high C/gamma |
| Decision Tree | `max_depth` | Overfitting (deep tree) |
| Random Forest | `n_estimators`, `max_features` | Slow training, correlated trees |
| Extra Trees | `max_features`, `bootstrap` | Similar to RF, but usually more variance reduction |
| AdaBoost | `n_estimators` + `learning_rate` | Overfitting with too many rounds & high LR |
| Gaussian NB | `var_smoothing` | Numerical instability on near-zero variance features |
| Multinomial NB | `alpha` | Overconfident predictions on rare feature values (low alpha) |
| Bernoulli NB | `alpha`, `binarize` | Wrong binarize threshold destroys signal |


## 🏆 Final Model Comparison

Let's collect every tuned classifier's test accuracy into one table + bar chart.

In [ ]:
results = {
    'Logistic Regression': grid_logreg.score(Xc_test_s, yc_test),
    'KNN': grid_knnc.score(Xc_test_s, yc_test),
    'SVC': grid_svc.score(Xc_test_s, yc_test),
    'Decision Tree': random_dtc.score(Xc_test, yc_test),
    'Random Forest': random_rfc.score(Xc_test, yc_test),
    'Extra Trees': random_etc.score(Xc_test, yc_test),
    'AdaBoost': grid_ada.score(Xc_test, yc_test),
    'Gaussian NB': grid_gnb.score(Xc_test_s, yc_test),
    'Multinomial NB': grid_mnb.score(Xc_test_mm, yc_test),
    'Bernoulli NB': grid_bnb.score(Xc_test_mm, yc_test),
}

results_df = pd.DataFrame(results.items(), columns=['Model', 'Test Accuracy']).sort_values('Test Accuracy', ascending=False)
display(results_df.style.background_gradient(cmap='Greens', subset=['Test Accuracy']))

import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
plt.barh(results_df['Model'], results_df['Test Accuracy'], color='#4C9A2A')
plt.xlabel('Test Accuracy')
plt.title('🏆 Tuned Classifier Comparison (Breast Cancer Dataset)')
plt.gca().invert_yaxis()
plt.xlim(0.85, 1.0)
plt.tight_layout()
plt.show()


## 📋 Cheat-Sheet: Tuning Strategy by Algorithm Family

| Family | Recommended search | Why |
|---|---|---|
| Linear models (Linear/Ridge/Lasso/Logistic) | `GridSearchCV` over `alpha`/`C` on a log scale | Small, cheap search space |
| Tree-based single tree | `RandomizedSearchCV` | Many interacting params |
| Ensembles (RF, Extra Trees, AdaBoost) | `RandomizedSearchCV` → refine with `GridSearchCV` | Large combinatorial space; random search explores faster, then narrow down |
| SVM/SVR | `GridSearchCV` with log-scaled `C`/`gamma` | Few params, but very sensitive — grid search is fine |
| KNN | `GridSearchCV` over small `k` range | Cheap to evaluate exhaustively |
| Naive Bayes | `GridSearchCV` over `alpha`/`var_smoothing` | Very few hyperparameters |

### 🎯 General best practices
1. **Always scale features** for distance/margin-based models (KNN, SVM, Logistic, Linear/Ridge/Lasso, NB with continuous features).
2. **Trees & ensembles don't need scaling.**
3. Use **log-scale** ranges for `alpha`, `C`, `gamma` (e.g. `[0.001, 0.01, 0.1, 1, 10, 100]`) — these parameters affect the model multiplicatively, not linearly.
4. Start with `RandomizedSearchCV` for big search spaces, then run `GridSearchCV` on the narrowed-down region.
5. Always use **cross-validation** (`cv=5` typical) — never tune directly against the test set.
6. Check `cv_results_` for variance across folds, not just the mean — a model that's "best on average" but unstable across folds may generalize worse.

---
### 🎓 You're done!
You now have a fully tunable reference for all 18 supervised ML algorithms. Swap in your own dataset by replacing the `Xr_*`/`yr_*` (regression) or `Xc_*`/`yc_*` (classification) variables at the top.
